# 01 Recalculate W/O landscape from replicate-level counts

This notebook rebuilds the core W/O genotype landscape tables from `data/processed/wo_counts.csv`.

The goal is to verify that the cleaned workflow reproduces the dissertation-associated processed outputs:

- `WO_genotype_counts_per_library_condition.csv`
- `WO_landscape_per_library.csv`

Statistical p-value calculations and low-read filtering sensitivity analyses are handled in later notebooks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Checkpoint output:", CHECKPOINT_DIR)

Project root: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape
Processed data: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/data/processed
Checkpoint output: /Users/jonathanfelix/Documents/Dissertation_GitHub/rbd-epistasis-landscape/outputs/checkpoints


In [2]:
wo_counts = pd.read_csv(PROCESSED_DIR / "wo_counts.csv")

expected_pooled = pd.read_csv(
    PROCESSED_DIR / "WO_genotype_counts_per_library_condition.csv"
)

expected_landscape = pd.read_csv(
    PROCESSED_DIR / "WO_landscape_per_library.csv"
)

print("wo_counts:", wo_counts.shape)
print("expected_pooled:", expected_pooled.shape)
print("expected_landscape:", expected_landscape.shape)

wo_counts: (5451, 6)
expected_pooled: (2304, 6)
expected_landscape: (768, 11)


In [3]:
required_columns = {
    "wo",
    "umi_count",
    "library_id",
    "condition",
    "replicate",
    "sample",
}

missing_columns = required_columns - set(wo_counts.columns)
assert not missing_columns, f"Missing required columns: {missing_columns}"

assert wo_counts["wo"].str.len().eq(8).all(), "All W/O genotypes should have length 8."
assert set(wo_counts["library_id"]).issubset({1, 2, 3}), "Unexpected library IDs."
assert set(wo_counts["condition"]).issubset({"Pre", "Pos", "Neg"}), "Unexpected conditions."
assert (wo_counts["umi_count"] >= 0).all(), "UMI counts should be non-negative."

print("Input validation passed.")

Input validation passed.


In [4]:
sample_coverage = (
    wo_counts
    .groupby(["sample", "library_id", "condition", "replicate"], as_index=False)
    .agg(
        total_umi_count=("umi_count", "sum"),
        n_detected_genotypes=("wo", "nunique")
    )
    .sort_values("total_umi_count")
)

sample_coverage

,sample,library_id,condition,replicate,total_umi_count,n_detected_genotypes
11,Library-2-Pos-2,2,Pos,2,440,155
20,Library-3-Pre-2,3,Pre,2,466,191
13,Library-2-Pre-2,2,Pre,2,2240,248
1,Library-1-Neg-2,1,Neg,2,13848,250
17,Library-3-Pos-1,3,Pos,1,14623,255
7,Library-1-Pre-2,1,Pre,2,89573,256
15,Library-3-Neg-2,3,Neg,2,242077,256
4,Library-1-Pos-2,1,Pos,2,338370,256
6,Library-1-Pre-1,1,Pre,1,3048475,256
16,Library-3-Neg-3,3,Neg,3,3426082,256


In [5]:
pooled = (
    wo_counts
    .groupby(["library_id", "condition", "wo"], as_index=False)
    .agg(umi_count=("umi_count", "sum"))
)

pooled["total_cond"] = (
    pooled
    .groupby(["library_id", "condition"])["umi_count"]
    .transform("sum")
)

pooled["freq"] = pooled["umi_count"] / pooled["total_cond"]

pooled = pooled.sort_values(["library_id", "condition", "wo"]).reset_index(drop=True)

pooled.to_csv(
    CHECKPOINT_DIR / "WO_genotype_counts_per_library_condition_recalculated.csv",
    index=False
)

pooled.head()

,library_id,condition,wo,umi_count,total_cond,freq
0,1,Neg,OOOOOOOO,267700,45002366,0.005949
1,1,Neg,OOOOOOOW,60615,45002366,0.001347
2,1,Neg,OOOOOOWO,156880,45002366,0.003486
3,1,Neg,OOOOOOWW,144912,45002366,0.003220
4,1,Neg,OOOOOWOO,30689,45002366,0.000682


In [6]:
expected_rows = 3 * 3 * 256

assert pooled.shape[0] == expected_rows, (
    f"Expected {expected_rows} pooled rows "
    "(3 libraries x 3 conditions x 256 genotypes), "
    f"observed {pooled.shape[0]}."
)

pooled_genotype_coverage = (
    pooled
    .groupby(["library_id", "condition"], as_index=False)
    .agg(
        n_genotypes=("wo", "nunique"),
        total_umi_count=("umi_count", "sum")
    )
    .sort_values(["library_id", "condition"])
)

assert pooled_genotype_coverage["n_genotypes"].eq(256).all(), (
    "Each library-condition group should contain 256 W/O genotypes."
)

pooled_genotype_coverage

,library_id,condition,n_genotypes,total_umi_count
0,1,Neg,256,45002366
1,1,Pos,256,26690098
2,1,Pre,256,31133073
3,2,Neg,256,26845007
4,2,Pos,256,53008735
5,2,Pre,256,15390714
6,3,Neg,256,3668159
7,3,Pos,256,15458642
8,3,Pre,256,29012677


In [7]:
pooled_compare = expected_pooled.merge(
    pooled,
    on=["library_id", "condition", "wo"],
    how="outer",
    suffixes=("_expected", "_recalculated"),
    indicator=True
)

print(pooled_compare["_merge"].value_counts())

for col in ["umi_count", "total_cond", "freq"]:
    expected_col = f"{col}_expected"
    recalculated_col = f"{col}_recalculated"
    max_abs_diff = (
        pooled_compare[expected_col] - pooled_compare[recalculated_col]
    ).abs().max()
    print(f"{col}: max absolute difference = {max_abs_diff}")

_merge
both          2304
left_only        0
right_only       0
Name: count, dtype: int64
umi_count: max absolute difference = 0
total_cond: max absolute difference = 0
freq: max absolute difference = 9.985502008591496e-17


In [8]:
landscape_freq = (
    pooled
    .pivot_table(
        index=["library_id", "wo"],
        columns="condition",
        values="freq",
        fill_value=0
    )
    .reset_index()
)

landscape_counts = (
    pooled
    .pivot_table(
        index=["library_id", "wo"],
        columns="condition",
        values="umi_count",
        fill_value=0
    )
    .reset_index()
    .rename(columns={
        "Pre": "count_pre",
        "Pos": "count_pos",
        "Neg": "count_neg",
    })
)

landscape_recalc = landscape_freq.merge(
    landscape_counts,
    on=["library_id", "wo"],
    how="left"
)

landscape_recalc.head()

condition,library_id,wo,Neg,Pos,Pre,count_neg,count_pos,count_pre
0,1,OOOOOOOO,0.005949,0.008597,0.003254,267700.0,229466.0,101311.0
1,1,OOOOOOOW,0.001347,0.001362,0.001796,60615.0,36354.0,55914.0
2,1,OOOOOOWO,0.003486,0.001391,0.002819,156880.0,37137.0,87776.0
3,1,OOOOOOWW,0.003220,0.000169,0.004767,144912.0,4517.0,148411.0
4,1,OOOOOWOO,0.000682,0.000340,0.000464,30689.0,9062.0,14458.0


In [9]:
epsilon = 1e-6

landscape_recalc["log2fc_pos_pre"] = np.log2(
    (landscape_recalc["Pos"] + epsilon) /
    (landscape_recalc["Pre"] + epsilon)
)

landscape_recalc["log2fc_neg_pre"] = np.log2(
    (landscape_recalc["Neg"] + epsilon) /
    (landscape_recalc["Pre"] + epsilon)
)

# Raw frequency shifts relative to the input/pre-selection population.
# These are descriptive absolute-frequency differences, not log enrichment values.
landscape_recalc["delta_pos"] = landscape_recalc["Pos"] - landscape_recalc["Pre"]
landscape_recalc["delta_neg"] = landscape_recalc["Neg"] - landscape_recalc["Pre"]

landscape_recalc["mut_count"] = landscape_recalc["wo"].str.count("O")

# Pos-vs-Neg enrichment contrast.
# This is calculated as the difference between Pos/Pre and Neg/Pre log2 fold-changes.
# Algebraically, this approximates log2(Pos/Neg), with Pre cancelling except for pseudocount effects.
landscape_recalc["ddelta"] = (
    landscape_recalc["log2fc_pos_pre"] - landscape_recalc["log2fc_neg_pre"]
)

landscape_recalc = landscape_recalc[
    [
        "library_id",
        "wo",
        "Neg",
        "Pos",
        "Pre",
        "log2fc_pos_pre",
        "log2fc_neg_pre",
        "delta_pos",
        "delta_neg",
        "mut_count",
        "ddelta",
        "count_pre",
        "count_pos",
        "count_neg",
    ]
].sort_values(["library_id", "wo"]).reset_index(drop=True)

landscape_recalc.to_csv(
    CHECKPOINT_DIR / "WO_landscape_per_library_recalculated.csv",
    index=False
)

landscape_recalc.head()

condition,library_id,wo,Neg,Pos,Pre,log2fc_pos_pre,log2fc_neg_pre,delta_pos,delta_neg,mut_count,ddelta,count_pre,count_pos,count_neg
0,1,OOOOOOOO,0.005949,0.008597,0.003254,1.401357,0.870073,0.005343,0.002694,8,0.531285,101311.0,229466.0,267700.0
1,1,OOOOOOOW,0.001347,0.001362,0.001796,-0.398696,-0.414820,-0.000434,-0.000449,7,0.016124,55914.0,36354.0,60615.0
2,1,OOOOOOWO,0.003486,0.001391,0.002819,-1.018301,0.306112,-0.001428,0.000667,7,-1.324413,87776.0,37137.0,156880.0
3,1,OOOOOOWW,0.003220,0.000169,0.004767,-4.807749,-0.565828,-0.004598,-0.001547,6,-4.241921,148411.0,4517.0,144912.0
4,1,OOOOOWOO,0.000682,0.000340,0.000464,-0.450684,0.553312,-0.000125,0.000218,7,-1.003995,14458.0,9062.0,30689.0


In [10]:
assert landscape_recalc.shape[0] == 3 * 256, (
    f"Expected {3 * 256} landscape rows "
    "(3 libraries x 256 genotypes), "
    f"observed {landscape_recalc.shape[0]}."
)

assert landscape_recalc["wo"].str.len().eq(8).all(), (
    "All W/O genotype strings should have length 8."
)

landscape_genotype_coverage = (
    landscape_recalc
    .groupby("library_id", as_index=False)
    .agg(
        n_genotypes=("wo", "nunique"),
        n_rows=("wo", "size")
    )
)

assert landscape_genotype_coverage["n_genotypes"].eq(256).all(), (
    "Each library should contain 256 W/O genotypes."
)

landscape_genotype_coverage

,library_id,n_genotypes,n_rows
0,1,256,256
1,2,256,256
2,3,256,256


In [11]:
expected_landscape_sorted = (
    expected_landscape
    .sort_values(["library_id", "wo"])
    .reset_index(drop=True)
)

landscape_compare = expected_landscape_sorted.merge(
    landscape_recalc,
    on=["library_id", "wo"],
    how="outer",
    suffixes=("_expected", "_recalculated"),
    indicator=True
)

print(landscape_compare["_merge"].value_counts())

compare_cols = [
    "Neg",
    "Pos",
    "Pre",
    "log2fc_pos_pre",
    "log2fc_neg_pre",
    "delta_pos",
    "delta_neg",
    "mut_count",
    "ddelta",
]

for col in compare_cols:
    expected_col = f"{col}_expected"
    recalculated_col = f"{col}_recalculated"
    max_abs_diff = (
        landscape_compare[expected_col] - landscape_compare[recalculated_col]
    ).abs().max()
    print(f"{col}: max absolute difference = {max_abs_diff}")

_merge
both          768
left_only       0
right_only      0
Name: count, dtype: int64
Neg: max absolute difference = 9.974659986866641e-17
Pos: max absolute difference = 9.985502008591496e-17
Pre: max absolute difference = 9.974659986866641e-17
log2fc_pos_pre: max absolute difference = 4.440892098500626e-16
log2fc_neg_pre: max absolute difference = 4.440892098500626e-16
delta_pos: max absolute difference = 9.985502008591496e-17
delta_neg: max absolute difference = 9.996344030316351e-17
mut_count: max absolute difference = 0
ddelta: max absolute difference = 4.440892098500626e-16


In [12]:
tolerance = 1e-9

pooled_freq_diff = (
    pooled_compare["freq_expected"] - pooled_compare["freq_recalculated"]
).abs().max()

landscape_numeric_diffs = {}

for col in [
    "Neg",
    "Pos",
    "Pre",
    "log2fc_pos_pre",
    "log2fc_neg_pre",
    "delta_pos",
    "delta_neg",
    "ddelta",
]:
    landscape_numeric_diffs[col] = (
        landscape_compare[f"{col}_expected"] -
        landscape_compare[f"{col}_recalculated"]
    ).abs().max()

print("Pooled frequency max difference:", pooled_freq_diff)
print("Landscape max differences:")
for col, diff in landscape_numeric_diffs.items():
    print(f"  {col}: {diff}")

if pooled_freq_diff < tolerance and all(diff < tolerance for diff in landscape_numeric_diffs.values()):
    print("\nClean recalculation reproduces expected pooled and landscape tables within tolerance.")
else:
    print("\nRecalculation differs from expected outputs. Review pseudocounts, sorting, or calculation conventions.")

Pooled frequency max difference: 9.985502008591496e-17
Landscape max differences:
  Neg: 9.974659986866641e-17
  Pos: 9.985502008591496e-17
  Pre: 9.974659986866641e-17
  log2fc_pos_pre: 4.440892098500626e-16
  log2fc_neg_pre: 4.440892098500626e-16
  delta_pos: 9.985502008591496e-17
  delta_neg: 9.996344030316351e-17
  ddelta: 4.440892098500626e-16

Clean recalculation reproduces expected pooled and landscape tables within tolerance.


## Recalculation result

The cleaned recalculation reproduces the archived dissertation-associated pooled count and W/O landscape tables within numerical floating-point tolerance.

Recovered calculation conventions:

- Frequencies are calculated from pooled UMI counts within each library-condition group.
- Log2 fold-change calculations use a pseudocount of `epsilon = 1e-6`.
- `delta_pos` and `delta_neg` are raw frequency shifts relative to the pre-selection input:
  - `delta_pos = Pos - Pre`
  - `delta_neg = Neg - Pre`
- `ddelta` is calculated as the Pos-vs-Neg log enrichment contrast:
  - `ddelta = log2fc_pos_pre - log2fc_neg_pre`

This confirms that `wo_counts.csv` is sufficient to reproduce the core dissertation-associated pooled count and W/O landscape tables.